# Stage 3 — Plan A: Temporal Heterogeneous Hypergraph (MELD)

## Architecture Summary
Identical to the IEMOCAP notebook but with MELD-specific settings:
- **Hidden dim**: 384 (vs 256 for IEMOCAP)
- **7 emotion classes**: neutral, joy, surprise, anger, sadness, disgust, fear
- **Evaluation**: fixed train/dev/test splits (not LOSO)
- **Missing audio**: `dia125_utt3` (train) and `dia110_utt7` (dev) zero-filled

## Model
**5 nodes/utterance**: text(RoBERTa-1024), audio(WavLM-1024), vis_begin/mid/end (AU⊕SigLIP2 = 1160)  
**4 hyperedge types**: Multimodal-utterance, Visual-arc, Contextual(×5), Speaker  
**Propagation**: Two-level attention, alternating inter/intra (K=4 layers)  
**Loss**: CB-Focal + λ·BCL

## Outputs
Saved to `Dataset/Processed/MELD/stage3_results/`:
- `best_model.pt` — best checkpoint (by dev WF1)
- `final_results.json` — test WF1/UAF1/ACC + config
- `class_distribution.png`, `training_curves.png`
- `confusion_matrix_test.png`, `per_class_f1_test.png`
- `tsne_representations.png`, `arc_attention_heatmap.png`

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json, warnings, time, copy
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.5 GB


In [2]:
# ============================================================
#  CONFIG  —  tune these before running
# ============================================================
PROJECT_ROOT = Path('/mnt/Work/ML/Thesis/BMVC/Hopeful')
FEAT_DIR     = PROJECT_ROOT / 'Dataset/Processed/MELD/features'
RESULTS_DIR  = PROJECT_ROOT / 'Dataset/Processed/MELD/stage3_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MELD_EMOTIONS = ['neutral', 'joy', 'surprise', 'anger', 'sadness', 'disgust', 'fear']
EMO2IDX       = {e: i for i, e in enumerate(MELD_EMOTIONS)}
NUM_CLASSES   = 7

# Input feature dimensions
D_TEXT   = 1024
D_AUDIO  = 1024
D_VIS    = 1160   # AU(8) + SigLIP2(1152)

# Architecture
HIDDEN   = 384    # larger than IEMOCAP (reduce to 256 for speed)
K_LAYERS = 4      # hypergraph layers (2 = faster)
W_PAST   = 4
W_FUTURE = 1
DROPOUT  = 0.3
CHUNK    = 64

# Training
EPOCHS        = 60
LR            = 2e-4
WEIGHT_DECAY  = 1e-4
LAMBDA_BCL    = 0.5
BETA_CB       = 0.9999
GAMMA_FOCAL   = 2.0
PATIENCE      = 12
WARMUP_EPOCHS = 5

# Known unplayable clips: text present, audio absent
MISSING_AUDIO = {
    'train': {'dia125_utt3'},
    'dev'  : {'dia110_utt7'},
    'test' : set(),
}

print(f'Config: HIDDEN={HIDDEN}, K_LAYERS={K_LAYERS}, W_PAST={W_PAST}, W_FUTURE={W_FUTURE}')
print(f'Results dir: {RESULTS_DIR}')

Config: HIDDEN=384, K_LAYERS=4, W_PAST=4, W_FUTURE=1
Results dir: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/stage3_results


In [3]:
# ============================================================
#  Load per-split features
# ============================================================
def load_split_feats(split):
    '''Load (text, audio, siglip2, openface_au) dicts for one MELD split.'''
    text  = torch.load(FEAT_DIR / f'text_roberta_large_{split}.pt',          weights_only=False)
    audio = torch.load(FEAT_DIR / f'audio_microsoft_wavlm_large_{split}.pt', weights_only=False)
    sig   = torch.load(FEAT_DIR / f'video_siglip2_temporal_{split}.pt',      weights_only=False)
    au    = torch.load(FEAT_DIR / f'video_openface_au_{split}.pt',           weights_only=False)
    return text, audio, sig, au


print('Loading MELD features (train / dev / test)...')
split_feats = {split: load_split_feats(split) for split in ['train', 'dev', 'test']}
print('  Loaded.')


def build_vis_absent(au_dict, sig_dict):
    '''Flag utterances where both AU and SigLIP2 are all-zero (no face).'''
    return {
        uid: (float(np.asarray(au_dict[uid]).sum())  == 0.0 and
              float(np.asarray(sig_dict[uid]).sum()) == 0.0)
        for uid in au_dict
    }


vis_absent_by_split = {
    split: build_vis_absent(split_feats[split][3], split_feats[split][2])
    for split in ['train', 'dev', 'test']
}

# Load labels
labels_df = pd.read_csv(PROJECT_ROOT / 'Dataset/Processed/MELD/labels.csv')
labels_df['label'] = labels_df['emotion'].map(EMO2IDX)

print(f'Total utterances: {len(labels_df)}')
print('\nEmotion distribution:')
print(labels_df['emotion'].value_counts())
print('\nSplit distribution:')
print(labels_df['split'].value_counts())

Loading MELD features (train / dev / test)...
  Loaded.
Total utterances: 13708

Emotion distribution:
emotion
neutral     6436
joy         2308
surprise    1636
anger       1607
sadness     1002
disgust      361
fear         358
Name: count, dtype: int64

Split distribution:
split
train    9989
test     2610
dev      1109
Name: count, dtype: int64


In [4]:
# ============================================================
#  Hypergraph incidence matrix constructor (same as IEMOCAP)
# ============================================================
def build_incidence_matrix(N, speakers, w_past=W_PAST, w_future=W_FUTURE):
    '''
    Build hypergraph incidence H: (5N, E) and edge_types: (E,).

    Node layout (5N total):
      text  [0,   N)   t_0 .. t_{N-1}
      audio [N,   2N)  a_0 .. a_{N-1}
      vis_b [2N,  3N)  vb_0 .. vb_{N-1}
      vis_m [3N,  4N)  vm_0 .. vm_{N-1}
      vis_e [4N,  5N)  ve_0 .. ve_{N-1}

    Edge types: 0=MM, 1=VisualArc, 2=Contextual, 3=Speaker
    '''
    unique_spk = list(dict.fromkeys(speakers))
    K       = len(unique_spk)
    spk2idx = {s: i for i, s in enumerate(unique_spk)}
    E       = 7 * N + 5 * K

    rows, cols = [], []
    modality_offsets = [0, N, 2*N, 3*N, 4*N]

    for i in range(N):
        for node in [i, N+i, 2*N+i, 3*N+i, 4*N+i]:
            rows.append(node); cols.append(i)
        for node in [2*N+i, 3*N+i, 4*N+i]:
            rows.append(node); cols.append(N + i)
        window = range(max(0, i - w_past), min(N, i + w_future + 1))
        for m_idx, offset in enumerate(modality_offsets):
            e_idx = 2*N + m_idx*N + i
            for j in window:
                rows.append(offset + j); cols.append(e_idx)
        k = spk2idx[speakers[i]]
        for m_idx, offset in enumerate(modality_offsets):
            rows.append(offset + i); cols.append(7*N + k*5 + m_idx)

    V      = 5 * N
    rows_t = torch.tensor(rows, dtype=torch.long)
    cols_t = torch.tensor(cols, dtype=torch.long)
    vals   = torch.ones(len(rows_t), dtype=torch.float32)
    H = torch.sparse_coo_tensor(
        torch.stack([rows_t, cols_t]), vals, (V, E)
    ).coalesce().to_dense()

    edge_types = torch.cat([
        torch.zeros(N,    dtype=torch.long),
        torch.ones(N,     dtype=torch.long),
        torch.full((5*N,), 2, dtype=torch.long),
        torch.full((5*K,), 3, dtype=torch.long),
    ])
    assert H.shape == (V, E) and edge_types.shape[0] == E
    return H, edge_types


# Quick check
N_v, spk_v = 6, ['Chandler', 'Ross', 'Monica', 'Chandler', 'Ross', 'Joey']
H_v, et_v  = build_incidence_matrix(N_v, spk_v)
print(f'H shape:    {H_v.shape}  (expected {5*N_v}, {7*N_v + 5*4})')
print(f'Edge types: {et_v.bincount().tolist()}  (MM, VA, Ctx, Spk)')
print('Incidence matrix OK.')

H shape:    torch.Size([30, 62])  (expected 30, 62)
Edge types: [6, 6, 30, 20]  (MM, VA, Ctx, Spk)
Incidence matrix OK.


In [5]:
# ============================================================
#  MELD dialog builder
# ============================================================
def build_meld_dialog(split, dia_id, sub_df):
    '''
    Build feature tensors + incidence matrix for one MELD dialog.

    Key differences from IEMOCAP:
    - clip_name is the utterance key (not utt_id)
    - Sort by integer utt_id (gaps possible due to cross-split fragmentation)
    - Missing audio clips -> zero-fill 1024-dim audio vector
    - Speaker scope is per-dialog (MELD has 304 global speakers)
    '''
    text_d, audio_d, sig_d, au_d = split_feats[split]
    miss   = MISSING_AUDIO[split]
    va_map = vis_absent_by_split[split]

    sub  = sub_df.sort_values('utt_id').reset_index(drop=True)
    utts = sub['clip_name'].tolist()
    N    = len(utts)

    # Text (always present)
    text_feat = torch.from_numpy(
        np.stack([np.asarray(text_d[u], dtype=np.float32) for u in utts]))

    # Audio — zero-fill known missing clips
    audio_list = []
    for u in utts:
        if u in miss or u not in audio_d:
            audio_list.append(np.zeros(D_AUDIO, dtype=np.float32))
        else:
            audio_list.append(np.asarray(audio_d[u], dtype=np.float32))
    audio_feat = torch.from_numpy(np.stack(audio_list))

    # Visual: concat AU(3,8) and SigLIP2(3,1152) -> (3, 1160)
    vis_list = []
    for u in utts:
        if u in au_d:
            au  = np.asarray(au_d[u],  dtype=np.float32)  # (3, 8)
            sig = np.asarray(sig_d[u], dtype=np.float32)  # (3, 1152)
            vis_list.append(np.concatenate([au, sig], axis=-1))
        else:
            vis_list.append(np.zeros((3, D_VIS), dtype=np.float32))
    vis_feat = torch.from_numpy(np.stack(vis_list))

    vis_absent = torch.tensor(
        [(u not in au_d) or va_map.get(u, False) for u in utts],
        dtype=torch.bool)

    labels_t = torch.tensor(sub['label'].tolist(), dtype=torch.long)
    speakers  = sub['speaker'].tolist()

    H_mat, et = build_incidence_matrix(N, speakers)

    return dict(
        dialog_id  = f'{split}_{dia_id}',
        utts       = utts,
        text       = text_feat,
        audio      = audio_feat,
        vis        = vis_feat,
        vis_absent = vis_absent,
        labels     = labels_t,
        speakers   = speakers,
        N          = N,
        H_mat      = H_mat,
        et         = et,
    )


def build_split_dialogs(split):
    '''Build list of dialog dicts for one MELD split.'''
    sub_df  = labels_df[labels_df['split'] == split]
    dialogs = []
    for dia_id, grp in sub_df.groupby('dia_id'):
        d = build_meld_dialog(split, dia_id, grp)
        if d['N'] >= 2:
            dialogs.append(d)
    return dialogs


print('Building dialog data for all splits...')
t0 = time.time()
train_dialogs = build_split_dialogs('train')
dev_dialogs   = build_split_dialogs('dev')
test_dialogs  = build_split_dialogs('test')
print(f'  Done in {time.time()-t0:.1f}s')
print(f'Train: {len(train_dialogs)} dialogs, '
      f'{sum(d["N"] for d in train_dialogs)} utterances')
print(f'Dev:   {len(dev_dialogs)} dialogs, '
      f'{sum(d["N"] for d in dev_dialogs)} utterances')
print(f'Test:  {len(test_dialogs)} dialogs, '
      f'{sum(d["N"] for d in test_dialogs)} utterances')

Building dialog data for all splits...
  Done in 0.8s
Train: 968 dialogs, 9919 utterances
Dev:   108 dialogs, 1103 utterances
Test:  268 dialogs, 2598 utterances


In [6]:
# ============================================================
#  Model components (identical to IEMOCAP, config-driven)
# ============================================================

class ModalityProjector(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)


class VisualProjector(nn.Module):
    def __init__(self, vis_in=1160, d=384, dropout=0.3):
        super().__init__()
        self.proj                = ModalityProjector(vis_in, d, dropout)
        self.visual_absent_embed = nn.Parameter(torch.zeros(3, d))

    def forward(self, vis_feat, vis_absent):
        h = self.proj(vis_feat)
        if vis_absent.any():
            absent_fill = self.visual_absent_embed.unsqueeze(0)
            mask        = vis_absent.view(-1, 1, 1).expand_as(h)
            h = torch.where(mask, absent_fill.expand_as(h), h)
        return h


class HypergraphConvLayer(nn.Module):
    def __init__(self, d, num_edge_types=4, dropout=0.3, chunk=64):
        super().__init__()
        self.d     = d
        self.chunk = chunk
        self.node_attn     = nn.Linear(d, 1)
        self.edge_type_emb = nn.Embedding(num_edge_types, d)
        self.hedge_attn    = nn.Linear(2 * d, 1)
        self.W_update      = nn.Linear(d, d)
        self.norm          = nn.LayerNorm(d)
        self.dropout       = nn.Dropout(dropout)

    def forward(self, X, H, edge_types):
        V, d = X.shape
        E    = H.shape[1]

        # Node -> Hyperedge
        node_scores = self.node_attn(X).expand(-1, E)
        node_scores = node_scores.masked_fill(H == 0, -1e9)
        alpha       = F.softmax(node_scores, dim=0) * H
        B           = alpha.T @ X

        # Hyperedge -> Node (chunked)
        type_emb       = self.edge_type_emb(edge_types)
        B_typed        = B + type_emb
        hedge_attn_mat = torch.zeros(V, E, device=X.device)

        for start in range(0, V, self.chunk):
            end     = min(start + self.chunk, V)
            C       = end - start
            x_chunk = X[start:end]
            x_exp   = x_chunk.unsqueeze(1).expand(-1, E, -1)
            b_exp   = B_typed.unsqueeze(0).expand(C, -1, -1)
            inp     = torch.cat([x_exp, b_exp], dim=-1)
            scores  = self.hedge_attn(inp).squeeze(-1)
            H_chunk = H[start:end]
            scores  = scores.masked_fill(H_chunk == 0, -1e9)
            hedge_attn_mat[start:end] = F.softmax(scores, dim=1) * H_chunk

        M     = hedge_attn_mat @ B
        X_new = self.norm(X + self.dropout(F.gelu(self.W_update(M))))
        return X_new


class PlanAModel(nn.Module):
    def __init__(self, d=384, num_classes=7, k_layers=4,
                 dropout=0.3, w_past=4, w_future=1, chunk=CHUNK):
        super().__init__()
        self.d        = d
        self.k_layers = k_layers
        self.w_past   = w_past
        self.w_future = w_future

        self.text_proj  = ModalityProjector(D_TEXT,  d, dropout)
        self.audio_proj = ModalityProjector(D_AUDIO, d, dropout)
        self.vis_proj   = VisualProjector(D_VIS, d, dropout)

        self.layers = nn.ModuleList([
            HypergraphConvLayer(d, num_edge_types=4, dropout=dropout, chunk=chunk)
            for _ in range(k_layers)
        ])

        self.arc_attn = nn.Linear(d, 1)

        self.fusion_mlp = nn.Sequential(
            nn.Linear(3 * d, d),
            nn.LayerNorm(d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, num_classes),
        )

        self.bcl_head = nn.Sequential(
            nn.Linear(3 * d, 256),
            nn.GELU(),
            nn.Linear(256, 128),
        )

    def forward(self, text, audio, vis, vis_absent, H_mat, edge_types, return_feats=False):
        N = text.shape[0]

        ht = self.text_proj(text)
        ha = self.audio_proj(audio)
        hv = self.vis_proj(vis, vis_absent)

        X = torch.cat([ht, ha, hv[:, 0], hv[:, 1], hv[:, 2]], dim=0)

        H  = H_mat.to(X.device)
        ET = edge_types.to(X.device)

        intra_mask = (ET == 1) | (ET == 2)
        inter_mask = (ET == 0) | (ET == 3)
        H_intra = H[:, intra_mask];  ET_intra = ET[intra_mask]
        H_inter = H[:, inter_mask];  ET_inter = ET[inter_mask]

        for li, layer in enumerate(self.layers):
            X = layer(X, H_inter, ET_inter) if li % 2 == 0 else layer(X, H_intra, ET_intra)

        ht_out  = X[0*N : 1*N]
        ha_out  = X[1*N : 2*N]
        hvb_out = X[2*N : 3*N]
        hvm_out = X[3*N : 4*N]
        hve_out = X[4*N : 5*N]

        vis_stack   = torch.stack([hvb_out, hvm_out, hve_out], dim=1)
        arc_scores  = self.arc_attn(vis_stack).squeeze(-1)
        arc_weights = F.softmax(arc_scores, dim=-1)
        hv_pooled   = (arc_weights.unsqueeze(-1) * vis_stack).sum(1)

        h_fused = torch.cat([ht_out, ha_out, hv_pooled], dim=-1)
        logits  = self.fusion_mlp(h_fused)

        if return_feats:
            feats = F.normalize(self.bcl_head(h_fused), dim=-1)
            return logits, feats, arc_weights
        return logits


# Sanity check
print('Sanity checking PlanAModel (MELD config)...')
_N  = 6
_sp = ['Chandler', 'Ross', 'Monica', 'Chandler', 'Ross', 'Joey']
_H, _et = build_incidence_matrix(_N, _sp)
_m  = PlanAModel(d=32, num_classes=NUM_CLASSES, k_layers=2, chunk=16).to(DEVICE)
_t  = torch.randn(_N, 1024).to(DEVICE)
_a  = torch.randn(_N, 1024).to(DEVICE)
_v  = torch.randn(_N, 3, 1160).to(DEVICE)
_ab = torch.zeros(_N, dtype=torch.bool).to(DEVICE)
_lg, _ft, _aw = _m(_t, _a, _v, _ab, _H, _et, return_feats=True)
print(f'  logits={_lg.shape}  feats={_ft.shape}  arc={_aw.shape}')
_lg.sum().backward()
print('  Backward: OK')
del _m, _t, _a, _v, _ab, _lg, _ft, _aw, _H, _et
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('PlanAModel OK.')

Sanity checking PlanAModel (MELD config)...
  logits=torch.Size([6, 7])  feats=torch.Size([6, 128])  arc=torch.Size([6, 3])
  Backward: OK
PlanAModel OK.


In [7]:
# ============================================================
#  Loss functions (same as IEMOCAP)
# ============================================================

class CBFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.9999, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        nc  = torch.tensor(class_counts, dtype=torch.float32)
        eff = (1.0 - beta ** nc) / (1.0 - beta)
        w   = (1.0 - beta) / eff
        w   = w / w.sum() * len(w)
        self.register_buffer('weights', w)

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, weight=self.weights, reduction='none')
        p_t   = torch.exp(-ce)
        focal = (1.0 - p_t) ** self.gamma * ce
        return focal.mean()


class BalancedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        N = features.shape[0]
        if N < 4 or labels.unique().numel() < 2:
            return torch.tensor(0.0, device=features.device, requires_grad=True)
        sim       = features @ features.T / self.temperature
        diag      = torch.eye(N, dtype=torch.bool, device=features.device)
        same_cls  = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask  = same_cls & ~diag
        n_pos     = pos_mask.sum(1, keepdim=True).float().clamp(min=1)
        log_denom = torch.logsumexp(sim.masked_fill(diag, -1e9), dim=1, keepdim=True)
        log_prob  = sim - log_denom
        loss      = -(pos_mask.float() * log_prob / n_pos).sum(1).mean()
        return loss


# Verify with MELD class distribution
meld_counts = [6436, 2308, 1636, 1607, 1002, 361, 358]  # MELD train counts
_cb = CBFocalLoss(meld_counts)
w_arr = _cb.weights.numpy()
print('CB-Focal weights for MELD:')
for emo, w in zip(MELD_EMOTIONS, w_arr):
    print(f'  {emo:<12} {w:.4f}')
print(f'(neutral={meld_counts[0]} -> smallest weight; fear={meld_counts[6]} -> largest)')
del _cb
print('Loss functions OK.')

CB-Focal weights for MELD:
  neutral      0.1687
  joy          0.3884
  surprise     0.5304
  anger        0.5392
  sadness      0.8396
  disgust      2.2576
  fear         2.2762
(neutral=6436 -> smallest weight; fear=358 -> largest)
Loss functions OK.


In [8]:
# ============================================================
#  Training utilities
# ============================================================

def compute_metrics(all_labels, all_preds):
    wf1     = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    uaf1    = f1_score(all_labels, all_preds, average='macro',    zero_division=0)
    acc     = accuracy_score(all_labels, all_preds)
    per_cls = f1_score(all_labels, all_preds, average=None,
                       zero_division=0, labels=list(range(NUM_CLASSES)))
    return {'wf1': float(wf1), 'uaf1': float(uaf1), 'acc': float(acc),
            'per_class': per_cls.tolist()}


def evaluate(model, dialogs, crit_focal, crit_bcl):
    model.eval()
    total_loss, all_preds, all_true = 0.0, [], []
    with torch.no_grad():
        for d in dialogs:
            if d['N'] < 2: continue
            text   = d['text'].to(DEVICE)
            audio  = d['audio'].to(DEVICE)
            vis    = d['vis'].to(DEVICE)
            absent = d['vis_absent'].to(DEVICE)
            labels = d['labels'].to(DEVICE)

            logits, feats, _ = model(
                text, audio, vis, absent, d['H_mat'], d['et'], return_feats=True)
            loss = crit_focal(logits, labels) + LAMBDA_BCL * crit_bcl(feats, labels)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(-1).cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_true, all_preds)
    metrics['loss'] = total_loss / max(len(dialogs), 1)
    return metrics, all_true, all_preds

In [9]:
# ============================================================
#  Training: train on train, monitor on dev, early stop on dev WF1
# ============================================================
all_train_labels = torch.cat([d['labels'] for d in train_dialogs])
class_counts     = [(all_train_labels == c).sum().item() for c in range(NUM_CLASSES)]
print('Train class distribution:')
for emo, cnt in zip(MELD_EMOTIONS, class_counts):
    print(f'  {emo:<12} {cnt}')

model = PlanAModel(
    d=HIDDEN, num_classes=NUM_CLASSES, k_layers=K_LAYERS,
    dropout=DROPOUT, w_past=W_PAST, w_future=W_FUTURE
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel parameters: {n_params:,}')

crit_focal = CBFocalLoss(class_counts, BETA_CB, GAMMA_FOCAL).to(DEVICE)
crit_bcl   = BalancedContrastiveLoss(0.07).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=1e-6)

history = {'train_loss': [], 'dev_loss': [],
           'train_wf1': [], 'dev_wf1': []}
best_wf1, best_state, no_improve = 0.0, None, 0
start_epoch = 0

resume_path = RESULTS_DIR / 'resume_checkpoint.pt'

# Skip training if final checkpoint already exists
if (RESULTS_DIR / 'best_model.pt').exists():
    print('\nbest_model.pt already exists — skipping training, loading checkpoint.')
    saved = torch.load(RESULTS_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
    model.load_state_dict(saved['state_dict'])
    history   = saved['history']
    best_wf1  = max(history['dev_wf1']) if history['dev_wf1'] else 0.0
    best_state = saved['state_dict']
else:
    # Resume from per-epoch checkpoint if available
    if resume_path.exists():
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        history     = ckpt['history']
        best_wf1    = ckpt['best_wf1']
        best_state  = ckpt['best_state']
        no_improve  = ckpt['no_improve']
        start_epoch = ckpt['epoch'] + 1
        print(f'\nResumed from epoch {start_epoch} (best dev WF1={best_wf1:.4f})')

    print(f'\nTraining MELD Plan A ({EPOCHS} epochs max, patience={PATIENCE})...')
    t_start = time.time()

    for epoch in range(start_epoch, EPOCHS):
        # Linear warmup
        if epoch < WARMUP_EPOCHS:
            for pg in optimizer.param_groups:
                pg['lr'] = LR * (epoch + 1) / WARMUP_EPOCHS

        model.train()
        train_loss, train_preds, train_true = 0.0, [], []
        order = list(range(len(train_dialogs)))
        np.random.shuffle(order)

        for di in order:
            d = train_dialogs[di]
            if d['N'] < 2: continue
            text   = d['text'].to(DEVICE)
            audio  = d['audio'].to(DEVICE)
            vis    = d['vis'].to(DEVICE)
            absent = d['vis_absent'].to(DEVICE)
            labels = d['labels'].to(DEVICE)

            optimizer.zero_grad()
            logits, feats, _ = model(
                text, audio, vis, absent, d['H_mat'], d['et'], return_feats=True)
            loss = crit_focal(logits, labels) + LAMBDA_BCL * crit_bcl(feats, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss  += loss.item()
            train_preds.extend(logits.argmax(-1).detach().cpu().tolist())
            train_true.extend(labels.cpu().tolist())

        if epoch >= WARMUP_EPOCHS:
            scheduler.step()

        dev_m, _, _  = evaluate(model, dev_dialogs, crit_focal, crit_bcl)
        train_m      = compute_metrics(train_true, train_preds)
        elapsed_min  = (time.time() - t_start) / 60

        history['train_loss'].append(train_loss / max(len(train_dialogs), 1))
        history['dev_loss'].append(dev_m['loss'])
        history['train_wf1'].append(train_m['wf1'])
        history['dev_wf1'].append(dev_m['wf1'])

        if (epoch + 1) % 10 == 0 or epoch < 3:
            print(f'  Epoch {epoch+1:3d}  '
                  f'train_wf1={train_m["wf1"]:.4f}  '
                  f'dev_wf1={dev_m["wf1"]:.4f}  '
                  f'loss={dev_m["loss"]:.4f}  '
                  f'[{elapsed_min:.1f}min]')

        if dev_m['wf1'] > best_wf1:
            best_wf1   = dev_m['wf1']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stop at epoch {epoch+1}')
                break

        # Save per-epoch resume checkpoint
        torch.save({
            'epoch'          : epoch,
            'model_state'    : model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'history'        : history,
            'best_wf1'       : best_wf1,
            'best_state'     : best_state,
            'no_improve'     : no_improve,
        }, resume_path)

    total_min = (time.time() - t_start) / 60
    print(f'\nTraining done.  Best dev WF1: {best_wf1:.4f}  '
          f'Total time: {total_min:.1f} min')

    # Remove resume checkpoint on clean completion
    if resume_path.exists():
        resume_path.unlink()

Train class distribution:
  neutral      4670
  joy          1730
  surprise     1196
  anger        1108
  sadness      679
  disgust      269
  fear         267

Model parameters: 2,616,336

Training MELD Plan A (60 epochs max, patience=12)...
  Epoch   1  train_wf1=0.1034  dev_wf1=0.1541  loss=1.3193  [0.1min]
  Epoch   2  train_wf1=0.1222  dev_wf1=0.1544  loss=1.2980  [0.2min]
  Epoch   3  train_wf1=0.1381  dev_wf1=0.0964  loss=1.2627  [0.3min]
  Epoch  10  train_wf1=0.3065  dev_wf1=0.2301  loss=1.3667  [1.1min]
  Epoch  20  train_wf1=0.5050  dev_wf1=0.3120  loss=1.8281  [2.2min]
  Epoch  30  train_wf1=0.6412  dev_wf1=0.3721  loss=2.2412  [3.3min]
  Epoch  40  train_wf1=0.7514  dev_wf1=0.4251  loss=2.5345  [4.5min]
  Epoch  50  train_wf1=0.8071  dev_wf1=0.4301  loss=2.6764  [5.6min]
  Epoch  60  train_wf1=0.8265  dev_wf1=0.4391  loss=2.6867  [6.7min]

Training done.  Best dev WF1: 0.4473  Total time: 6.7 min


In [10]:
# ============================================================
#  Evaluate best model on TEST set
# ============================================================
model.load_state_dict(best_state)
torch.save({
    'state_dict': best_state,
    'history'   : history,
    'config'    : {'hidden': HIDDEN, 'k_layers': K_LAYERS,
                   'w_past': W_PAST, 'w_future': W_FUTURE},
}, RESULTS_DIR / 'best_model.pt')

print('Evaluating on TEST set...')
test_m, test_true, test_preds = evaluate(model, test_dialogs, crit_focal, crit_bcl)

print(f'\nTest WF1  : {test_m["wf1"]:.4f}')
print(f'Test UAF1 : {test_m["uaf1"]:.4f}')
print(f'Test ACC  : {test_m["acc"]:.4f}')
print('\nPer-class F1 (test):')
for emo, f1 in zip(MELD_EMOTIONS, test_m['per_class']):
    print(f'  {emo:<12} {f1:.4f}')

# Collect arc weights + reps from test set
arc_by_label  = {c: [] for c in range(NUM_CLASSES)}
reps_list, reps_labels = [], []
model.eval()
with torch.no_grad():
    for d in test_dialogs:
        if d['N'] < 2: continue
        _, feats, arc_w = model(
            d['text'].to(DEVICE),  d['audio'].to(DEVICE),
            d['vis'].to(DEVICE),   d['vis_absent'].to(DEVICE),
            d['H_mat'], d['et'],   return_feats=True)
        reps_list.append(feats.cpu().numpy())
        reps_labels.extend(d['labels'].tolist())
        for c in range(NUM_CLASSES):
            mask = d['labels'] == c
            if mask.any():
                arc_by_label[c].append(arc_w[mask].cpu().numpy())

test_reps   = np.concatenate(reps_list, axis=0)
test_rlabs  = np.array(reps_labels)

# Save final JSON
result_save = {
    'test_metrics': test_m,
    'dev_best_wf1': best_wf1,
    'config'      : {'hidden': HIDDEN, 'k_layers': K_LAYERS,
                     'w_past': W_PAST, 'w_future': W_FUTURE,
                     'lambda_bcl': LAMBDA_BCL, 'lr': LR, 'epochs': EPOCHS},
    'history'     : history,
}
with open(RESULTS_DIR / 'final_results.json', 'w') as f:
    json.dump(result_save, f, indent=2)
print('\nSaved: best_model.pt, final_results.json')

Evaluating on TEST set...

Test WF1  : 0.4494
Test UAF1 : 0.3114
Test ACC  : 0.4392

Per-class F1 (test):
  neutral      0.5660
  joy          0.3864
  surprise     0.3433
  anger        0.3978
  sadness      0.2997
  disgust      0.0820
  fear         0.1043

Saved: best_model.pt, final_results.json


In [11]:
# ============================================================
#  Visualization 0: MELD class distribution (context for imbalance)
# ============================================================
split_counts = {}
for split in ['train', 'dev', 'test']:
    sub = labels_df[labels_df['split'] == split]
    split_counts[split] = [int((sub['emotion'] == e).sum()) for e in MELD_EMOTIONS]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: grouped bar
x = np.arange(NUM_CLASSES)
w = 0.25
colors_split = ['steelblue', 'darkorange', 'forestgreen']
for i, (split, cnts) in enumerate(split_counts.items()):
    axes[0].bar(x + (i-1) * w, cnts, w, label=split, color=colors_split[i], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(MELD_EMOTIONS, rotation=20, fontsize=10)
axes[0].set_ylabel('Count'); axes[0].set_title('MELD Emotion Counts by Split')
axes[0].legend(fontsize=10); axes[0].grid(axis='y', alpha=0.3)

# Right: log-scale bar (shows long-tail more clearly)
total_cnt = [sum(split_counts[s][ci] for s in ['train','dev','test'])
             for ci in range(NUM_CLASSES)]
axes[1].bar(x, total_cnt, color='mediumpurple', alpha=0.85)
axes[1].set_yscale('log')
axes[1].set_xticks(x)
axes[1].set_xticklabels(MELD_EMOTIONS, rotation=20, fontsize=10)
axes[1].set_ylabel('Count (log scale)')
axes[1].set_title('MELD Total Emotion Distribution (log)')
for ci, cnt in enumerate(total_cnt):
    axes[1].text(ci, cnt * 1.1, str(cnt), ha='center', va='bottom', fontsize=8)

plt.suptitle('MELD Dataset Class Distribution\n(neutral dominant, fear/disgust minority)',
             fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

Saved: class_distribution.png


In [12]:
# ============================================================
#  Visualization 1: Training curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], color='steelblue',  label='train', lw=1.8)
axes[0].plot(ep, history['dev_loss'],   color='darkorange', label='dev',   lw=1.8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves (train / dev)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_wf1'], color='steelblue',  label='train', lw=1.8)
axes[1].plot(ep, history['dev_wf1'],   color='darkorange', label='dev',   lw=1.8)
best_ep = int(np.argmax(history['dev_wf1'])) + 1
best_v  = max(history['dev_wf1'])
axes[1].axvline(best_ep, color='red', ls=':', lw=1.5, alpha=0.8,
                label=f'Best epoch={best_ep} (WF1={best_v:.4f})')
axes[1].axhline(test_m['wf1'], color='green', ls='--', lw=1.5, alpha=0.8,
                label=f'Test WF1={test_m["wf1"]:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('WF1')
axes[1].set_title('WF1 Curves (train / dev / test)')
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle(f'MELD Plan A — Training Curves  '
             f'(Best dev WF1={best_wf1:.4f}  Test WF1={test_m["wf1"]:.4f})',
             fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

Saved: training_curves.png


In [13]:
# ============================================================
#  Visualization 2: Confusion matrix (test set)
# ============================================================
cm   = confusion_matrix(test_true, test_preds, labels=list(range(NUM_CLASSES)))
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Normalized
sns.heatmap(cm_n, ax=axes[0],
            xticklabels=MELD_EMOTIONS, yticklabels=MELD_EMOTIONS,
            annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            annot_kws={'size': 9}, linewidths=0.5)
axes[0].set_title('Normalized Confusion Matrix (test)', fontsize=11)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].tick_params(labelsize=9, rotation=30)

# Raw counts
sns.heatmap(cm, ax=axes[1],
            xticklabels=MELD_EMOTIONS, yticklabels=MELD_EMOTIONS,
            annot=True, fmt='d', cmap='Oranges',
            annot_kws={'size': 8}, linewidths=0.5)
axes[1].set_title('Raw Count Confusion Matrix (test)', fontsize=11)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].tick_params(labelsize=9, rotation=30)

plt.suptitle(f'MELD Plan A — Confusion Matrix  '
             f'(WF1={test_m["wf1"]:.4f}  UAF1={test_m["uaf1"]:.4f})', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_test.png')

Saved: confusion_matrix_test.png


In [14]:
# ============================================================
#  Visualization 3: Per-class F1 bar chart (test)
# ============================================================
fig, ax = plt.subplots(figsize=(11, 5))
x       = np.arange(NUM_CLASSES)
per_cls = test_m['per_class']
palette = plt.cm.tab10(np.linspace(0, 0.7, NUM_CLASSES))

bars = ax.bar(x, per_cls, color=palette, alpha=0.9)
ax.axhline(test_m['wf1'],  color='black', ls='--', lw=1.5,
           label=f'WF1={test_m["wf1"]:.4f}')
ax.axhline(test_m['uaf1'], color='gray',  ls=':',  lw=1.5,
           label=f'UAF1={test_m["uaf1"]:.4f}')

for bi, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.012,
            f'{per_cls[bi]:.3f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(MELD_EMOTIONS, fontsize=11)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('MELD Plan A — Per-Class F1 Score (Test Set)', fontsize=12)
ax.legend(fontsize=10); ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Annotate class sizes
test_sub = labels_df[labels_df['split'] == 'test']
for ci, emo in enumerate(MELD_EMOTIONS):
    n_test = int((test_sub['emotion'] == emo).sum())
    ax.text(ci, -0.06, f'n={n_test}', ha='center', va='top', fontsize=7.5, color='gray')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_f1_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_class_f1_test.png')

Saved: per_class_f1_test.png


In [15]:
# ============================================================
#  Visualization 4: t-SNE of test utterance representations
# ============================================================
reps  = test_reps
rlabs = test_rlabs

if len(reps) > 3000:
    idx   = np.random.choice(len(reps), 3000, replace=False)
    reps  = reps[idx]
    rlabs = rlabs[idx]

print(f'Running t-SNE on {len(reps)} points (128-dim)...')
tsne = TSNE(n_components=2, perplexity=40, random_state=42,
            max_iter=1000, learning_rate='auto', init='pca')
X_2d = tsne.fit_transform(reps)
print('t-SNE done.')

palette = plt.cm.tab10(np.linspace(0, 0.7, NUM_CLASSES))
fig, ax = plt.subplots(figsize=(10, 7))
for ci, emo in enumerate(MELD_EMOTIONS):
    mask = rlabs == ci
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[palette[ci]], label=f'{emo} (n={mask.sum()})',
               alpha=0.55, s=12, linewidths=0)
ax.legend(markerscale=2.5, fontsize=9, loc='best', framealpha=0.85)
ax.set_title('MELD Plan A — t-SNE of Test Utterance Representations\n'
             '(BCL head, 128-dim, colored by emotion)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tsne_representations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: tsne_representations.png')

Running t-SNE on 2598 points (128-dim)...
t-SNE done.
Saved: tsne_representations.png


In [16]:
# ============================================================
#  Visualization 5: Visual arc attention heatmap (test set)
# ============================================================
arc_mean = np.zeros((NUM_CLASSES, 3))
arc_std  = np.zeros((NUM_CLASSES, 3))

for ci in range(NUM_CLASSES):
    segs = arc_by_label[ci]
    if segs:
        stacked      = np.concatenate(segs, axis=0)
        arc_mean[ci] = stacked.mean(0)
        arc_std[ci]  = stacked.std(0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: heatmap
im = axes[0].imshow(arc_mean, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(['Begin', 'Mid', 'End'], fontsize=11)
axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_yticklabels(MELD_EMOTIONS, fontsize=10)
axes[0].set_title('Mean Arc Attention Weight\nby Emotion Class (test)', fontsize=11)
axes[0].set_xlabel('Temporal Segment')
axes[0].set_ylabel('Emotion')
for ci in range(NUM_CLASSES):
    for ti in range(3):
        clr = 'white' if arc_mean[ci, ti] > 0.55 else 'black'
        axes[0].text(ti, ci, f'{arc_mean[ci,ti]:.3f}',
                     ha='center', va='center', fontsize=9, color=clr)
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Right: stacked bar per emotion
x  = np.arange(NUM_CLASSES)
w0 = arc_mean[:, 0]; w1 = arc_mean[:, 1]; w2 = arc_mean[:, 2]
axes[1].bar(x, w0,        label='Begin', color='#2196F3', alpha=0.9)
axes[1].bar(x, w1, w0,    label='Mid',   color='#FF9800', alpha=0.9)
axes[1].bar(x, w2, w0+w1, label='End',   color='#4CAF50', alpha=0.9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(MELD_EMOTIONS, fontsize=10, rotation=15)
axes[1].set_ylabel('Attention Weight')
axes[1].set_ylim(0, 1)
axes[1].set_title('Arc Attention Distribution\nper Emotion Class (test)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('MELD Plan A — Visual Arc Attention (Expression Arc Analysis)',
             fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'arc_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: arc_attention_heatmap.png')
print('Interpretation: deviations from uniform (0.333) indicate arc carries emotion signal.')

Saved: arc_attention_heatmap.png
Interpretation: deviations from uniform (0.333) indicate arc carries emotion signal.


In [17]:
# ============================================================
#  Final confirmation
# ============================================================
print('=' * 60)
print('MELD Plan A — Complete')
print('=' * 60)
print(f'\nResults directory: {RESULTS_DIR}')
saved_files = sorted(RESULTS_DIR.iterdir())
for fp in saved_files:
    size_kb = fp.stat().st_size / 1024
    print(f'  {fp.name:<44} {size_kb:>8.1f} KB')

print(f'\nTest WF1  : {test_m["wf1"]:.4f}')
print(f'Test UAF1 : {test_m["uaf1"]:.4f}')
print(f'Test ACC  : {test_m["acc"]:.4f}')
print('\nComparison targets:')
print('  MM-DFN  baseline  WF1: ~58.17  (beat this first)')
print('  GraphSmile TPAMI  WF1: ~60-62  (competitive range)')
print('  HRG-SSA IJCAI-25  WF1: ~65+    (current ceiling)')

MELD Plan A — Complete

Results directory: /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/stage3_results
  arc_attention_heatmap.png                       125.9 KB
  best_model.pt                                 10243.9 KB
  class_distribution.png                           77.8 KB
  confusion_matrix_test.png                       152.1 KB
  final_results.json                                6.7 KB
  per_class_f1_test.png                            56.9 KB
  training_curves.png                             113.5 KB
  tsne_representations.png                        228.1 KB

Test WF1  : 0.4494
Test UAF1 : 0.3114
Test ACC  : 0.4392

Comparison targets:
  MM-DFN  baseline  WF1: ~58.17  (beat this first)
  GraphSmile TPAMI  WF1: ~60-62  (competitive range)
  HRG-SSA IJCAI-25  WF1: ~65+    (current ceiling)
